In [1]:
from arcengine import GameAction
from arc import MyArcSession
from arc_agi import OperationMode, Arcade
import io
import logging
import numpy as np
from dotenv import load_dotenv
from PIL import Image
from sandbox import SandboxOrchestrator
from google.genai import types
from agent import JackAgent

load_dotenv()
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

PALETTE = np.array([
    [0xFF, 0xFF, 0xFF], [0xCC, 0xCC, 0xCC], [
        0x99, 0x99, 0x99], [0x66, 0x66, 0x66],
    [0x33, 0x33, 0x33], [0x00, 0x00, 0x00], [
        0xE5, 0x3A, 0xA3], [0xFF, 0x7B, 0xCC],
    [0xF9, 0x3C, 0x31], [0x1E, 0x93, 0xFF], [
        0x88, 0xD8, 0xF1], [0xFF, 0xDC, 0x00],
    [0xFF, 0x85, 0x1B], [0x92, 0x12, 0x31], [
        0x4F, 0xCC, 0x30], [0xA3, 0x56, 0xD6],
], dtype=np.uint8)

PRICING = {
    "gemini-3.1-flash-lite-preview":  {"in": 0.25, "in_200k": 0.25, "out": 1.5, "out_200k": 1.5},
    "gemini-3-flash-preview":  {"in": 0.50, "in_200k": 0.50, "out": 3.00, "out_200k": 3.00},
    "gemini-3.1-pro-preview":  {"in": 2.00, "in_200k": 4.00, "out": 12.00, "out_200k": 18.00},
}

# Nano Banana 2 (image gen) — $0.50/1M input, $60/1M output tokens
IMAGE_PRICING = {"in": 0.50, "out": 60.00}


def render_frame(frame: np.ndarray, size: int = 192) -> Image.Image:
    rgb = PALETTE[np.clip(np.asarray(frame, dtype=np.uint8), 0, 15)]
    return Image.fromarray(rgb).resize((size, size), Image.NEAREST)


def estimate_cost(usage, model):
    p = PRICING.get(model, PRICING["gemini-3-flash-preview"])
    ctx = usage["prompt_tokens"]
    rate_in = p["in_200k"] if ctx > 200_000 else p["in"]
    rate_out = p["out_200k"] if ctx > 200_000 else p["out"]
    cost_in = usage["total_prompt_tokens"] * rate_in / 1_000_000
    cost_out = (usage["output_tokens"] +
                usage["thinking_tokens"]) * rate_out / 1_000_000
    cost_img = (usage.get("image_prompt_tokens", 0) * IMAGE_PRICING["in"] +
                usage.get("image_output_tokens", 0) * IMAGE_PRICING["out"]) / 1_000_000
    return cost_in + cost_out + cost_img


# --- setup (fresh container + game on every run) ---
arcade = Arcade(operation_mode=OperationMode.NORMAL.value)
scorecard_id: str = arcade.open_scorecard(tags=["jackagent"])
arc_session = MyArcSession(
    game_id=["ls20", "ft09", "vc33"][0], arcade=arcade, scorecard_id=scorecard_id
)
sbx = SandboxOrchestrator()
agent = JackAgent(sbx=sbx, arc_session=arc_session)


2026-04-14 21:43:22 | INFO | Successfully fetched 25 environment(s) from API


21:43:22 Successfully fetched 25 environment(s) from API


2026-04-14 21:43:22 | INFO | Created new scorecard: bd53e8e8-0230-48e3-a191-62df588fb2de


21:43:22 Created new scorecard: bd53e8e8-0230-48e3-a191-62df588fb2de


2026-04-14 21:43:22 | INFO | Successfully fetched metadata for game ls20


21:43:22 Successfully fetched metadata for game ls20


2026-04-14 21:43:22 | INFO | Recording to recordings/bd53e8e8-0230-48e3-a191-62df588fb2de/ls20-9607627b-df82fa93-145b-4983-84e7-b796b7a45a4a.jsonl


21:43:22 Recording to recordings/bd53e8e8-0230-48e3-a191-62df588fb2de/ls20-9607627b-df82fa93-145b-4983-84e7-b796b7a45a4a.jsonl


2026-04-14 21:43:22 | INFO | Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py


21:43:22 Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py


In [2]:
# # solved = 3 * [GameAction.ACTION3] + 4 * [GameAction.ACTION1] + 3 * [GameAction.ACTION4] + 3 * [GameAction.ACTION1]
# import hashlib

# seen = {}


# def write_state():
#     pass


# loop = 3 * [GameAction.ACTION3] + 3 * [GameAction.ACTION1] + \
#     [GameAction.ACTION4, GameAction.ACTION2,
#         GameAction.ACTION3, GameAction.ACTION1, GameAction.ACTION1]

# for a in loop:
#     frame_hash = hashlib.md5(arc_session.obs.frame[-1].tobytes()).hexdigest()
#     print(frame_hash)
#     if frame_hash in seen:
#         print("been in this state before")
#     arc_session.do_action(a)
#     seen[frame_hash] = 1
#     display(render_frame(arc_session.obs.frame[-1]))
#     arc_session.reset()

# for a in loop:
#     frame_hash = hashlib.md5(arc_session.obs.frame[-1].tobytes()).hexdigest()
#     print(frame_hash)
#     if frame_hash in seen:
#         print("been in this state before")
#     arc_session.do_action(a)
#     seen[frame_hash] = 1
#     display(render_frame(arc_session.obs.frame[-1]))
#     arc_session.reset()

In [3]:
print(arc_session.obs)

{
  "game_id": "ls20-9607627b",
  "state": "NOT_FINISHED",
  "levels_completed": 0,
  "win_levels": 7,
  "action_input": {
    "id": 0,
    "data": {},
    "reasoning": null
  },
  "guid": "df82fa93-145b-4983-84e7-b796b7a45a4a",
  "full_reset": true,
  "available_actions": [
    1,
    2,
    3,
    4
  ]
}


In [ ]:

# # --- kick off ---
# agent.contents = [
#     types.Content(role="user", parts=[types.Part(text=(
#         f"Game: {arc_session.obs.game_id} | "
#         f"Score: {arc_session.obs.levels_completed}/{arc_session.obs.win_levels} | "
#         f"Actions: {[GameAction.from_id(a).name for a in arc_session.obs.available_actions]}\n\n"
#         f"Render the board, look at it, figure out the rules, solve the game."
#     ))])
# ]

import io

# pre-render the first frame to PNG bytes so we can attach it inline
_first_frame_img = render_frame(arc_session.obs.frame[-1], size=512)
_buf = io.BytesIO()
_first_frame_img.save(_buf, format="PNG")
_first_frame_png = _buf.getvalue()

agent.contents = [
    types.Content(role="user", parts=[
        types.Part(text=(
            "We're playing a 2D puzzle game together.\n\n"
            f"Game: {arc_session.obs.game_id} | "
            f"Score: {arc_session.obs.levels_completed}/{arc_session.obs.win_levels} | "
            f"Actions available: {[GameAction.from_id(a).name for a in arc_session.obs.available_actions]}\n\n"
            "You can either CLICK places (ACTION6 with x,y) or use up/down/left/right arrows (ACTION1-4). "
            "Games vary in logic and you don't know the rules — you figure them out by experimenting, "
            "taking notes about the board, and identifying where things are.\n\n"
            "Attached is the first frame of this game (64x64 grid upscaled to 512x512).\n\n"
            "**Your first task:** Look at this frame carefully. Then call `generate_image` with a prompt "
            "that asks for a simplified, annotated version of this exact frame — you can simplify "
            "details but the core shapes, positions, and structure must stay the same. Ask it to label "
            "the distinct objects/regions you see (e.g. 'player', 'goal', 'wall', 'item A', etc.) so you "
            "can reason about them by name.\n\n"
            "**Then:** Use the annotated image you generated as your mental map. Form a hypothesis about "
            "the game's rules. Take actions to test your hypothesis. Update your notes. Solve the game.\n\n"
            "Remember: `render_board` renders the current state from state.json, `view` lets you look at "
            "any image in the sandbox, `take_action` submits a move."
        )),
        types.Part(inline_data=types.Blob(
            mime_type="image/png",
            data=_first_frame_png,
        )),
    ])
]


def render_tool_result(name, args, result, agent, image_bytes=None):
    """Render a single tool result inline."""
    if name == "take_action":
        action = args.get("action", "")
        xy = f" ({args.get('x')},{args.get('y')})" if action == "ACTION6" else ""
        print(f"    {action}{xy}  {result[:200]}")
        obs = agent.arc_session.obs
        if obs.frame:
            display(render_frame(obs.frame[-1]))
    elif name == "view":
        path = args.get("file_path", "")
        if image_bytes:
            print(f"    view: {path}")
            display(Image.open(io.BytesIO(image_bytes)).resize(
                (192, 192), Image.NEAREST))
        else:
            print(f"    view: {path}  ({result.count(chr(10))+1} lines)")
    elif name == "bash":
        print(f"    $ {args.get('command', '')}")
        for line in result[:300].splitlines():
            print(f"      {line}")
    elif name == "generate_image":
        prompt = args.get("prompt", "")
        print(f"    generate_image: {prompt[:150]}")
        if image_bytes:
            display(Image.open(io.BytesIO(image_bytes)).resize(
                (256, 256), Image.NEAREST))
        print(f"    {result[:200]}")
    else:
        print(f"    {name}: {result[:200]}")


def render_turn(agent):
    """Walk assistant parts in order; each function_call is followed by its matching response."""
    calls = agent.generate_response()
    has_calls = bool(calls)
    assistant_content = agent.contents[-2] if has_calls else agent.contents[-1]

    # build id -> (result_str, image_bytes) from the function_response turn
    id_to_result: dict = {}
    id_to_bytes: dict = {}
    if has_calls:
        for fr_part in agent.contents[-1].parts:
            fr = fr_part.function_response
            if not fr:
                continue
            try:
                id_to_result[fr.id] = fr.response.get("result", "")
            except Exception:
                id_to_result[fr.id] = ""
            if fr.parts:
                for inner in fr.parts:
                    if inner.inline_data and inner.inline_data.data:
                        id_to_bytes[fr.id] = inner.inline_data.data
                        break

    # fallback iterator in case id matching fails
    call_iter = iter(calls)

    for p in assistant_content.parts:
        if hasattr(p, "text") and p.text:
            print(p.text[:2000])
        if p.function_call:
            fc = p.function_call
            args = dict(fc.args) if fc.args else {}
            args_str = ", ".join(f"{k}={v}" for k, v in args.items())
            print(f"  -> {fc.name}({args_str})")
            result = id_to_result.get(fc.id)
            if result is None:
                try:
                    c = next(call_iter)
                    result = str(c["result"])
                except StopIteration:
                    result = ""
            img_bytes = id_to_bytes.get(fc.id)
            render_tool_result(fc.name, args, str(result), agent, image_bytes=img_bytes)

    return calls


# --- loop ---
for turn in range(300):
    u = agent.usage
    cost = estimate_cost(u, agent.model)
    obs = agent.arc_session.obs
    state = obs.state.name

    print(f"\n{'─' * 70}")
    print(
        f"  turn {turn+1}  score={obs.levels_completed}/{obs.win_levels}  state={state}  ${cost:.4f}")
    print(f"{'─' * 70}")

    render_turn(agent)

    final_state = agent.arc_session.obs.state.name
    if final_state in ("WIN", "GAME_OVER"):
        print(f"\n{'=' * 70}")
        print(f"  {final_state} in {turn+1} turns    ${cost:.4f} total")
        print(f"{'=' * 70}")
        break
